In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Notebook is inside /notebooks, so go one level up to the project root.
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "m5" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "m5" / "processed"

print("Raw data folder:", RAW_DIR.resolve())
print("Processed data folder:", PROCESSED_DIR.resolve())

# Show exactly what files we downloaded before loading anything.
print("\nFiles in raw folder:")
for file in sorted(RAW_DIR.iterdir()):
    print(f"  {file.name}")

In [ ]:
# Check the size of every raw M5 file before loading them into memory.
for file in sorted(RAW_DIR.glob("*.csv")):
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name:<30} {size_mb:>8.2f} MB")

In [ ]:
# Load the three core M5 datasets.
# We are deliberately NOT loading the validation or submission files yet.
sales = pd.read_csv(RAW_DIR / "sales_train_evaluation.csv")
calendar = pd.read_csv(RAW_DIR / "calendar.csv")
prices = pd.read_csv(RAW_DIR / "sell_prices.csv")

print("Sales shape:   ", sales.shape)
print("Calendar shape:", calendar.shape)
print("Prices shape:  ", prices.shape)

In [ ]:
# ============================================================
# INSPECT THE THREE CORE M5 TABLES
# ============================================================

print("SALES COLUMNS:")
print(sales.columns[:20].tolist())

print("\nSALES SAMPLE:")
display(sales.iloc[:3, :15])

print("\nCALENDAR COLUMNS:")
print(calendar.columns.tolist())

print("\nCALENDAR SAMPLE:")
display(calendar.head(3))

print("\nPRICE COLUMNS:")
print(prices.columns.tolist())

print("\nPRICE SAMPLE:")
display(prices.head(3))

In [ ]:
# ============================================================
# BASIC M5 DATASET AUDIT
# ============================================================

day_cols = [col for col in sales.columns if col.startswith("d_")]

print("Number of item-store series:", len(sales))
print("Number of daily sales columns:", len(day_cols))

print("\nUnique items:", sales["item_id"].nunique())
print("Unique stores:", sales["store_id"].nunique())
print("Unique states:", sales["state_id"].nunique())
print("Unique departments:", sales["dept_id"].nunique())
print("Unique categories:", sales["cat_id"].nunique())

print("\nFirst sales day:", day_cols[0])
print("Last sales day:", day_cols[-1])

print("\nDuplicate series IDs:", sales["id"].duplicated().sum())

print("\nMissing values:")
print("Sales:", sales.isna().sum().sum())
print("Calendar:", calendar.isna().sum().sum())
print("Prices:", prices.isna().sum().sum())

print("\nCalendar columns:")
print(calendar.columns.tolist())

In [ ]:
# ============================================================
# SALES ACTIVITY / ZERO-DEMAND AUDIT
# ============================================================

sales_values = sales[day_cols]

total_values = sales_values.size
zero_values = (sales_values == 0).sum().sum()
positive_values = (sales_values > 0).sum().sum()
negative_values = (sales_values < 0).sum().sum()

print("Total daily sales observations:", f"{total_values:,}")
print("Zero observations:", f"{zero_values:,}",
      f"({zero_values / total_values:.2%})")
print("Positive observations:", f"{positive_values:,}",
      f"({positive_values / total_values:.2%})")
print("Negative observations:", f"{negative_values:,}")

# For each item-store series, find the first and last day with positive sales.
has_positive_sales = sales_values.gt(0)

first_positive_position = has_positive_sales.values.argmax(axis=1)

# Series that never record a positive sale need to be identified separately.
never_positive = ~has_positive_sales.any(axis=1)

first_positive_day = pd.Series(
    first_positive_position + 1,
    index=sales.index
)

first_positive_day[never_positive] = np.nan

print("\nSeries with no positive sales:", never_positive.sum())

print("\nFirst positive-sale day distribution:")
print(first_positive_day.describe())

print("\nSeries whose first positive sale occurs after:")
for day in [30, 90, 180, 365, 730, 1000]:
    count = (first_positive_day > day).sum()
    print(f"d_{day:<4}: {count:>6,} series ({count / len(sales):.2%})")

In [ ]:
# ============================================================
# ITEM AVAILABILITY AUDIT
# Compare first recorded price with first positive sale
# ============================================================

# First week in which each store-item combination has a recorded price.
first_price_week = (
    prices.groupby(["store_id", "item_id"])["wm_yr_wk"]
    .min()
    .reset_index(name="first_price_week")
)

# Calendar tells us which daily M5 day belongs to each Walmart week.
day_to_week = calendar[["d", "wm_yr_wk"]].copy()

# Extract numeric day number: d_1 -> 1, d_1941 -> 1941.
day_to_week["day_num"] = (
    day_to_week["d"]
    .str.replace("d_", "", regex=False)
    .astype(int)
)

# Find the first calendar day belonging to each Walmart week.
week_first_day = (
    day_to_week.groupby("wm_yr_wk")["day_num"]
    .min()
    .reset_index(name="first_price_day")
)

first_price = first_price_week.merge(
    week_first_day,
    left_on="first_price_week",
    right_on="wm_yr_wk",
    how="left"
)

# Build one row per sales series.
availability_audit = sales[
    ["id", "store_id", "item_id"]
].copy()

availability_audit["first_positive_day"] = first_positive_day

availability_audit = availability_audit.merge(
    first_price[
        ["store_id", "item_id", "first_price_week", "first_price_day"]
    ],
    on=["store_id", "item_id"],
    how="left"
)

availability_audit["days_price_before_first_sale"] = (
    availability_audit["first_positive_day"]
    - availability_audit["first_price_day"]
)

print("Series:", len(availability_audit))

print(
    "Series without any recorded price:",
    availability_audit["first_price_week"].isna().sum()
)

print("\nFirst price day:")
print(availability_audit["first_price_day"].describe())

print("\nDays from first recorded price to first positive sale:")
print(
    availability_audit["days_price_before_first_sale"].describe()
)

print("\nSample:")
display(availability_audit.head(10))

In [ ]:
# ============================================================
# INVESTIGATE PRICE-TO-FIRST-SALE OUTLIERS
# ============================================================

gap = availability_audit["days_price_before_first_sale"]

print("Gap distribution:")
for days in [0, 7, 14, 30, 90, 180, 365]:
    count = (gap > days).sum()
    print(
        f"More than {days:>3} days: "
        f"{count:>6,} series ({count / len(availability_audit):.2%})"
    )

print("\nLargest gaps:")
display(
    availability_audit
    .sort_values("days_price_before_first_sale", ascending=False)
    .head(20)
)

In [ ]:
# ============================================================
# ZERO-DEMAND RATE DURING ACTIVE ITEM-STORE PERIODS
# ============================================================

active_zero_count = 0
active_positive_count = 0
active_observation_count = 0
pre_active_observation_count = 0

sales_array = sales[day_cols].to_numpy()

for i, first_day in enumerate(availability_audit["first_price_day"]):
    # Convert M5 day number (d_1 = 1) to Python's zero-based index.
    start_idx = int(first_day) - 1

    series_active_sales = sales_array[i, start_idx:]

    active_observation_count += len(series_active_sales)
    pre_active_observation_count += start_idx

    active_zero_count += np.sum(series_active_sales == 0)
    active_positive_count += np.sum(series_active_sales > 0)

print("Total observations:", f"{sales_array.size:,}")

print(
    "Pre-availability observations:",
    f"{pre_active_observation_count:,}",
    f"({pre_active_observation_count / sales_array.size:.2%})"
)

print(
    "Active observations:",
    f"{active_observation_count:,}",
    f"({active_observation_count / sales_array.size:.2%})"
)

print(
    "\nActive zero-demand observations:",
    f"{active_zero_count:,}",
    f"({active_zero_count / active_observation_count:.2%})"
)

print(
    "Active positive-demand observations:",
    f"{active_positive_count:,}",
    f"({active_positive_count / active_observation_count:.2%})"
)

In [ ]:
# ============================================================
# PRICE AVAILABILITY AFTER PRODUCT ACTIVATION
# ============================================================

# Number of distinct weeks in which each store-item has a price.
price_summary = (
    prices.groupby(["store_id", "item_id"])
    .agg(
        first_price_week=("wm_yr_wk", "min"),
        last_price_week=("wm_yr_wk", "max"),
        weeks_with_price=("wm_yr_wk", "nunique")
    )
    .reset_index()
)

# Give every Walmart week an ordered position:
# first week = 0, next week = 1, etc.
all_weeks = (
    calendar[["wm_yr_wk"]]
    .drop_duplicates()
    .sort_values("wm_yr_wk")
    .reset_index(drop=True)
)

all_weeks["week_position"] = np.arange(len(all_weeks))

week_position_map = dict(
    zip(all_weeks["wm_yr_wk"], all_weeks["week_position"])
)

# Convert first/last week into positions so we can calculate
# how many weeks should exist between them.
price_summary["first_week_position"] = (
    price_summary["first_price_week"].map(week_position_map)
)

price_summary["last_week_position"] = (
    price_summary["last_price_week"].map(week_position_map)
)

price_summary["expected_weeks"] = (
    price_summary["last_week_position"]
    - price_summary["first_week_position"]
    + 1
)

# Missing weeks inside the active price period.
price_summary["missing_price_weeks"] = (
    price_summary["expected_weeks"]
    - price_summary["weeks_with_price"]
)

print("Store-item series:", len(price_summary))

print(
    "Series with at least one internal missing-price week:",
    (price_summary["missing_price_weeks"] > 0).sum()
)

print("\nMissing-price-week distribution:")
print(price_summary["missing_price_weeks"].describe())

print("\nSeries with the most missing price weeks:")
display(
    price_summary
    .sort_values("missing_price_weeks", ascending=False)
    .head(10)
)

In [ ]:
# ============================================================
# CHECK WHETHER PRICE AVAILABILITY ENDS BEFORE THE DATASET ENDS
# ============================================================

last_dataset_week = calendar.loc[
    calendar["d"] == day_cols[-1], "wm_yr_wk"
].iloc[0]

print("Last sales day:", day_cols[-1])
print("Last dataset week:", last_dataset_week)

ended_early = price_summary[
    price_summary["last_price_week"] < last_dataset_week
]

print(
    "\nSeries whose price history ends before the final sales week:",
    len(ended_early),
    f"({len(ended_early) / len(price_summary):.2%})"
)

print("\nLast-price-week distribution:")
print(price_summary["last_price_week"].describe())

print("\nExamples ending earliest:")
display(
    price_summary
    .sort_values("last_week_position")
    .head(10)
)

In [ ]:
# ============================================================
# CALENDAR INTEGRITY AUDIT
# ============================================================

calendar_check = calendar.copy()
calendar_check["date"] = pd.to_datetime(calendar_check["date"])

print("Calendar rows:", len(calendar_check))
print("Date range:",
      calendar_check["date"].min(),
      "to",
      calendar_check["date"].max())

print("\nDuplicate dates:",
      calendar_check["date"].duplicated().sum())

print("Duplicate M5 day IDs:",
      calendar_check["d"].duplicated().sum())

# Check whether every date follows the previous date continuously.
date_gaps = calendar_check["date"].sort_values().diff().dt.days

print("Non-consecutive date gaps:",
      (date_gaps.dropna() != 1).sum())

# Check whether every sales day d_1 ... d_1941 exists in calendar.csv.
sales_days = set(day_cols)
calendar_days = set(calendar_check["d"])

missing_sales_days = sales_days - calendar_days

print("\nSales days missing from calendar:",
      len(missing_sales_days))

# Examine missing values column-by-column.
print("\nMissing values by calendar column:")
print(calendar_check.isna().sum())

# Examine possible values for event types and SNAP.
print("\nEvent type 1 values:")
print(calendar_check["event_type_1"].value_counts(dropna=False))

print("\nEvent type 2 values:")
print(calendar_check["event_type_2"].value_counts(dropna=False))

print("\nSNAP values:")
for col in ["snap_CA", "snap_TX", "snap_WI"]:
    print(f"\n{col}:")
    print(calendar_check[col].value_counts(dropna=False))

In [ ]:
# ============================================================
# DAILY DEMAND SUMMARY FOR CALENDAR/EVENT ANALYSIS
# ============================================================

# Total units sold across all item-store series for each M5 day.
daily_total_sales = sales[day_cols].sum(axis=0)

daily_demand = pd.DataFrame({
    "d": daily_total_sales.index,
    "total_sales": daily_total_sales.values
})

# Attach calendar information to each sales day.
daily_demand = daily_demand.merge(
    calendar_check,
    on="d",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(daily_demand))
print("Missing calendar matches:", daily_demand["date"].isna().sum())

display(
    daily_demand[
        [
            "date",
            "d",
            "total_sales",
            "weekday",
            "month",
            "event_name_1",
            "event_type_1",
            "snap_CA",
            "snap_TX",
            "snap_WI",
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# BASIC DEMAND SEASONALITY
# ============================================================

# Average total demand by weekday.
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekday_sales = (
    daily_demand.groupby("weekday")["total_sales"]
    .agg(["mean", "median", "std", "count"])
    .reindex(weekday_order)
)

print("AVERAGE DEMAND BY WEEKDAY:")
display(weekday_sales.round(2))


# Average total demand by month.
monthly_sales = (
    daily_demand.groupby("month")["total_sales"]
    .agg(["mean", "median", "std", "count"])
)

print("\nAVERAGE DEMAND BY MONTH:")
display(monthly_sales.round(2))


# Relative weekday effect.
overall_daily_mean = daily_demand["total_sales"].mean()

weekday_sales["vs_overall_%"] = (
    (weekday_sales["mean"] / overall_daily_mean - 1) * 100
)

print("\nWEEKDAY EFFECT RELATIVE TO OVERALL DAILY AVERAGE:")
display(weekday_sales[["mean", "vs_overall_%"]].round(2))

In [ ]:
# ============================================================
# EVENT / HOLIDAY DEMAND EFFECT
# ============================================================

event_rows = daily_demand[
    daily_demand["event_name_1"].notna()
].copy()

print("Number of event days:", len(event_rows))
print("Unique events:", event_rows["event_name_1"].nunique())

print("\nEvent types:")
print(event_rows["event_type_1"].value_counts())


# ------------------------------------------------------------
# Compare event days with ordinary days
# ------------------------------------------------------------

daily_demand["is_event"] = daily_demand["event_name_1"].notna()

event_comparison = (
    daily_demand.groupby("is_event")["total_sales"]
    .agg(["mean", "median", "std", "count"])
)

print("\nEVENT VS NON-EVENT DAYS:")
display(event_comparison.round(2))


# ------------------------------------------------------------
# Demand around events: -3 days through +3 days
# ------------------------------------------------------------

event_indices = daily_demand.index[daily_demand["is_event"]]

event_window_results = []

for offset in range(-3, 4):

    valid_indices = event_indices + offset

    # Keep only positions that actually exist in the dataset.
    valid_indices = valid_indices[
        (valid_indices >= 0) &
        (valid_indices < len(daily_demand))
    ]

    values = daily_demand.loc[valid_indices, "total_sales"]

    event_window_results.append({
        "days_from_event": offset,
        "mean_sales": values.mean(),
        "median_sales": values.median(),
        "count": len(values)
    })

event_window = pd.DataFrame(event_window_results)

print("\nDEMAND AROUND EVENTS:")
display(event_window.round(2))


# ------------------------------------------------------------
# Demand by event type
# ------------------------------------------------------------

event_type_sales = (
    event_rows.groupby("event_type_1")["total_sales"]
    .agg(["mean", "median", "std", "count"])
    .sort_values("mean", ascending=False)
)

print("\nDEMAND BY EVENT TYPE:")
display(event_type_sales.round(2))

In [ ]:
# ============================================================
# EVENT EFFECT AFTER CONTROLLING FOR WEEKDAY
# ============================================================

# Calculate normal average demand for each weekday.
weekday_baseline = (
    daily_demand.loc[~daily_demand["is_event"]]
    .groupby("weekday")["total_sales"]
    .mean()
)

# Expected sales based purely on weekday.
daily_demand["weekday_baseline"] = (
    daily_demand["weekday"].map(weekday_baseline)
)

# Percentage above/below the normal level for that weekday.
daily_demand["weekday_adjusted_pct"] = (
    (daily_demand["total_sales"] / daily_demand["weekday_baseline"] - 1)
    * 100
)

# ------------------------------------------------------------
# Examine -3 to +3 days around every event.
# ------------------------------------------------------------

event_indices = daily_demand.index[daily_demand["is_event"]]

adjusted_event_results = []

for offset in range(-3, 4):

    valid_indices = event_indices + offset

    valid_indices = valid_indices[
        (valid_indices >= 0) &
        (valid_indices < len(daily_demand))
    ]

    values = daily_demand.loc[
        valid_indices,
        "weekday_adjusted_pct"
    ]

    adjusted_event_results.append({
        "days_from_event": offset,
        "mean_adjusted_%": values.mean(),
        "median_adjusted_%": values.median(),
        "count": len(values)
    })

adjusted_event_window = pd.DataFrame(adjusted_event_results)

print("EVENT EFFECT AFTER CONTROLLING FOR WEEKDAY:")
display(adjusted_event_window.round(2))


# ------------------------------------------------------------
# Event-day effect by event type.
# ------------------------------------------------------------

event_type_adjusted = (
    daily_demand[daily_demand["is_event"]]
    .groupby("event_type_1")["weekday_adjusted_pct"]
    .agg(["mean", "median", "std", "count"])
    .sort_values("mean", ascending=False)
)

print("\nEVENT TYPE EFFECT AFTER CONTROLLING FOR WEEKDAY:")
display(event_type_adjusted.round(2))

In [ ]:
# ============================================================
# INDIVIDUAL EVENT EFFECTS
# ============================================================

event_effect_rows = []

# Analyze every unique primary event.
for event_name in daily_demand["event_name_1"].dropna().unique():

    event_indices = daily_demand.index[
        daily_demand["event_name_1"] == event_name
    ]

    row = {
        "event_name": event_name,
        "occurrences": len(event_indices)
    }

    # Examine two days before through one day after.
    for offset in [-2, -1, 0, 1]:

        valid_indices = event_indices + offset

        valid_indices = valid_indices[
            (valid_indices >= 0) &
            (valid_indices < len(daily_demand))
        ]

        values = daily_demand.loc[
            valid_indices,
            "weekday_adjusted_pct"
        ]

        # Use simple, predictable column names.
        column_name = {
            -2: "two_days_before_%",
            -1: "one_day_before_%",
             0: "event_day_%",
             1: "one_day_after_%"
        }[offset]

        row[column_name] = values.mean()

    event_effect_rows.append(row)


individual_event_effects = (
    pd.DataFrame(event_effect_rows)
    .sort_values("event_day_%", ascending=False)
    .reset_index(drop=True)
)

print("INDIVIDUAL EVENT EFFECTS")
print("(percentage above/below normal demand for the same weekday)")

display(individual_event_effects.round(2))

In [ ]:
# ============================================================
# EVENT EFFECTS BY PRODUCT CATEGORY
# ============================================================

category_daily_parts = []

for category in sales["cat_id"].unique():

    # Select every item-store series belonging to this category.
    category_mask = sales["cat_id"] == category

    # Aggregate sales across those series for each day.
    category_totals = sales.loc[category_mask, day_cols].sum(axis=0)

    temp = pd.DataFrame({
        "d": day_cols,
        "category": category,
        "category_sales": category_totals.values
    })

    category_daily_parts.append(temp)

category_daily = pd.concat(
    category_daily_parts,
    ignore_index=True
)

# Attach calendar information.
category_daily = category_daily.merge(
    calendar_check[
        ["d", "date", "weekday", "event_name_1", "event_type_1"]
    ],
    on="d",
    how="left",
    validate="many_to_one"
)

category_daily["is_event"] = (
    category_daily["event_name_1"].notna()
)


# ------------------------------------------------------------
# Create a normal weekday baseline separately for each category.
# ------------------------------------------------------------

category_weekday_baseline = (
    category_daily[~category_daily["is_event"]]
    .groupby(["category", "weekday"])["category_sales"]
    .mean()
    .rename("weekday_baseline")
    .reset_index()
)

category_daily = category_daily.merge(
    category_weekday_baseline,
    on=["category", "weekday"],
    how="left"
)

category_daily["weekday_adjusted_pct"] = (
    (
        category_daily["category_sales"]
        / category_daily["weekday_baseline"]
    ) - 1
) * 100


# ------------------------------------------------------------
# Analyze selected major events by category.
# ------------------------------------------------------------

major_events = [
    "SuperBowl",
    "Easter",
    "Mother's day",
    "Father's day",
    "IndependenceDay",
    "LaborDay",
    "Halloween",
    "Thanksgiving",
    "Christmas",
    "NewYear"
]

results = []

for category in category_daily["category"].unique():

    category_data = (
        category_daily[category_daily["category"] == category]
        .reset_index(drop=True)
    )

    for event in major_events:

        event_indices = category_data.index[
            category_data["event_name_1"] == event
        ]

        if len(event_indices) == 0:
            continue

        row = {
            "category": category,
            "event": event,
            "occurrences": len(event_indices)
        }

        for offset in [-2, -1, 0, 1]:

            valid_indices = event_indices + offset

            valid_indices = valid_indices[
                (valid_indices >= 0) &
                (valid_indices < len(category_data))
            ]

            values = category_data.loc[
                valid_indices,
                "weekday_adjusted_pct"
            ]

            column_name = {
                -2: "two_days_before_%",
                -1: "one_day_before_%",
                 0: "event_day_%",
                 1: "one_day_after_%"
            }[offset]

            row[column_name] = values.mean()

        results.append(row)


category_event_effects = pd.DataFrame(results)

display(
    category_event_effects
    .sort_values(["event", "category"])
    .round(2)
)

In [ ]:
# ============================================================
# SNAP EFFECT BY STATE AND PRODUCT CATEGORY
# ============================================================

snap_daily_parts = []

# Build daily demand for every state × category combination.
for state in sales["state_id"].unique():

    for category in sales["cat_id"].unique():

        mask = (
            (sales["state_id"] == state) &
            (sales["cat_id"] == category)
        )

        daily_totals = sales.loc[mask, day_cols].sum(axis=0)

        temp = pd.DataFrame({
            "d": day_cols,
            "state": state,
            "category": category,
            "sales": daily_totals.values
        })

        snap_daily_parts.append(temp)


snap_daily = pd.concat(
    snap_daily_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# Attach calendar information.
# ------------------------------------------------------------

snap_daily = snap_daily.merge(
    calendar_check[
        [
            "d",
            "date",
            "weekday",
            "snap_CA",
            "snap_TX",
            "snap_WI"
        ]
    ],
    on="d",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# Each state should use ONLY its own SNAP indicator.
# ------------------------------------------------------------

snap_daily["snap_active"] = 0

for state in ["CA", "TX", "WI"]:

    state_mask = snap_daily["state"] == state

    snap_daily.loc[state_mask, "snap_active"] = (
        snap_daily.loc[state_mask, f"snap_{state}"]
    )


# ------------------------------------------------------------
# Build weekday baselines separately for every
# state × category combination.
# ------------------------------------------------------------

weekday_baseline = (
    snap_daily
    .groupby(["state", "category", "weekday"])["sales"]
    .mean()
    .rename("weekday_baseline")
    .reset_index()
)

snap_daily = snap_daily.merge(
    weekday_baseline,
    on=["state", "category", "weekday"],
    how="left"
)


# Percentage difference from normal demand for that weekday.
snap_daily["weekday_adjusted_pct"] = (
    (snap_daily["sales"] / snap_daily["weekday_baseline"]) - 1
) * 100


# ------------------------------------------------------------
# Compare SNAP vs non-SNAP days.
# ------------------------------------------------------------

snap_effect = (
    snap_daily
    .groupby(["state", "category", "snap_active"])
    ["weekday_adjusted_pct"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
)

print("SNAP EFFECT BY STATE AND CATEGORY")
print("(percentage relative to normal demand for the same weekday)")

display(
    snap_effect.round(2)
)

In [ ]:
# ============================================================
# PRICE INTEGRITY AND VARIATION AUDIT
# ============================================================

print("Price rows:", f"{len(prices):,}")

print("\nMissing prices:")
print(prices["sell_price"].isna().sum())

print("\nZero or negative prices:")
print((prices["sell_price"] <= 0).sum())

print("\nOverall price distribution:")
print(prices["sell_price"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))


# ------------------------------------------------------------
# Check duplicate store-item-week records.
# ------------------------------------------------------------

duplicate_price_keys = prices.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

print("\nDuplicate store-item-week price records:",
      duplicate_price_keys)


# ------------------------------------------------------------
# Measure price variation within each item-store series.
# ------------------------------------------------------------

series_price_stats = (
    prices.groupby(["store_id", "item_id"])["sell_price"]
    .agg(
        min_price="min",
        max_price="max",
        mean_price="mean",
        unique_prices="nunique"
    )
    .reset_index()
)

series_price_stats["price_range_pct"] = (
    (
        series_price_stats["max_price"]
        - series_price_stats["min_price"]
    )
    / series_price_stats["mean_price"]
) * 100


print("\nUnique-price-count distribution per item-store:")
print(series_price_stats["unique_prices"].describe())


print("\nSeries that never change price:",
      (series_price_stats["unique_prices"] == 1).sum(),
      f"({(series_price_stats['unique_prices'] == 1).mean():.2%})")


print("\nWithin-series price-range % distribution:")
print(
    series_price_stats["price_range_pct"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


print("\nLargest price variation:")
display(
    series_price_stats
    .sort_values("price_range_pct", ascending=False)
    .head(10)
    .round(2)
)

In [ ]:
# ============================================================
# INVESTIGATE EXTREME PRICE VARIATION
# ============================================================

extreme_series = (
    series_price_stats
    .sort_values("price_range_pct", ascending=False)
    .head(10)[["store_id", "item_id"]]
)

extreme_price_history = prices.merge(
    extreme_series,
    on=["store_id", "item_id"],
    how="inner"
)

extreme_price_history = extreme_price_history.sort_values(
    ["store_id", "item_id", "wm_yr_wk"]
)

# Show every distinct price and how many weeks it occurred.
extreme_price_levels = (
    extreme_price_history
    .groupby(["store_id", "item_id", "sell_price"])
    .agg(
        first_week=("wm_yr_wk", "min"),
        last_week=("wm_yr_wk", "max"),
        weeks=("wm_yr_wk", "count")
    )
    .reset_index()
    .sort_values(["store_id", "item_id", "sell_price"])
)

print("PRICE LEVELS FOR THE 10 MOST EXTREME SERIES:")
display(extreme_price_levels)


# Also inspect how common very large price ranges are.
print("\nNUMBER OF SERIES ABOVE DIFFERENT PRICE-RANGE THRESHOLDS:")

for threshold in [25, 50, 100, 200, 500]:
    count = (series_price_stats["price_range_pct"] > threshold).sum()

    print(
        f"> {threshold:>3}% range: "
        f"{count:>5,} series "
        f"({count / len(series_price_stats):.2%})"
    )

In [ ]:
# ============================================================
# PRICE CHANGE / DISCOUNT VS WEEKLY DEMAND
# ============================================================

# ------------------------------------------------------------
# 1. Convert daily sales into weekly sales.
# ------------------------------------------------------------

sales_id_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

# Map each sales day (d_1, d_2, ...) to its M5 week.
day_to_week = (
    calendar_check[
        calendar_check["d"].isin(day_cols)
    ][["d", "wm_yr_wk"]]
    .set_index("d")["wm_yr_wk"]
)

# Group the daily sales columns by week.
# Transposing here lets us aggregate d_1...d_1941 into weeks
# without first creating a huge daily long-format table.
weekly_sales_values = (
    sales[day_cols]
    .T
    .groupby(day_to_week)
    .sum()
    .T
)

weekly_sales_values.columns.name = None

# Convert the weekly matrix to long format.
weekly_sales = (
    weekly_sales_values
    .assign(
        item_id=sales["item_id"].values,
        dept_id=sales["dept_id"].values,
        cat_id=sales["cat_id"].values,
        store_id=sales["store_id"].values,
        state_id=sales["state_id"].values
    )
    .melt(
        id_vars=sales_id_cols,
        var_name="wm_yr_wk",
        value_name="weekly_sales"
    )
)


# ------------------------------------------------------------
# 2. Join weekly demand with weekly prices.
#
# Inner join intentionally keeps only weeks where the item
# has a recorded price — our previously frozen active-period
# definition.
# ------------------------------------------------------------

weekly_price_demand = weekly_sales.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="inner",
    validate="one_to_one"
)

print("Active item-store-week observations:",
      f"{len(weekly_price_demand):,}")


# ------------------------------------------------------------
# 3. Calculate each series' typical price.
#
# Median is used because it is less distorted by rare extreme
# prices such as the $107 example we just investigated.
# ------------------------------------------------------------

typical_price = (
    weekly_price_demand
    .groupby(["store_id", "item_id"])["sell_price"]
    .median()
    .rename("typical_price")
    .reset_index()
)

weekly_price_demand = weekly_price_demand.merge(
    typical_price,
    on=["store_id", "item_id"],
    how="left",
    validate="many_to_one"
)

weekly_price_demand["price_vs_typical_pct"] = (
    (
        weekly_price_demand["sell_price"]
        / weekly_price_demand["typical_price"]
    ) - 1
) * 100


# ------------------------------------------------------------
# 4. Put price changes into interpretable buckets.
# ------------------------------------------------------------

weekly_price_demand["price_bucket"] = pd.cut(
    weekly_price_demand["price_vs_typical_pct"],
    bins=[
        -np.inf,
        -20,
        -10,
        -5,
        5,
        10,
        20,
        np.inf
    ],
    labels=[
        ">20% cheaper",
        "10-20% cheaper",
        "5-10% cheaper",
        "Typical ±5%",
        "5-10% higher",
        "10-20% higher",
        ">20% higher"
    ]
)


# ------------------------------------------------------------
# 5. Compare weekly demand against each series' own
# typical weekly demand.
# ------------------------------------------------------------

typical_weekly_demand = (
    weekly_price_demand
    .groupby(["store_id", "item_id"])["weekly_sales"]
    .median()
    .rename("typical_weekly_sales")
    .reset_index()
)

weekly_price_demand = weekly_price_demand.merge(
    typical_weekly_demand,
    on=["store_id", "item_id"],
    how="left",
    validate="many_to_one"
)

# Avoid percentage division by zero for extremely intermittent
# products whose median weekly sales are zero.
valid_price_analysis = weekly_price_demand[
    weekly_price_demand["typical_weekly_sales"] > 0
].copy()

valid_price_analysis["demand_vs_typical_pct"] = (
    (
        valid_price_analysis["weekly_sales"]
        / valid_price_analysis["typical_weekly_sales"]
    ) - 1
) * 100


# ------------------------------------------------------------
# 6. Summarize price-demand relationship.
# ------------------------------------------------------------

price_demand_summary = (
    valid_price_analysis
    .groupby(
        ["cat_id", "price_bucket"],
        observed=True
    )["demand_vs_typical_pct"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

print("\nPRICE VS DEMAND BY CATEGORY")
print("Demand change is relative to each item's typical weekly demand.")

display(
    price_demand_summary.round(2)
)

In [ ]:
# ============================================================
# SALES SPIKE / OUTLIER AUDIT
# ============================================================

# Use the active weekly dataset we already created in Cell 22.
# This avoids treating pre-availability periods as zero demand.

weekly_series_stats = (
    weekly_price_demand
    .groupby(["store_id", "item_id"])["weekly_sales"]
    .agg(
        mean_weekly_sales="mean",
        median_weekly_sales="median",
        std_weekly_sales="std",
        max_weekly_sales="max"
    )
    .reset_index()
)

# ------------------------------------------------------------
# Global distribution of active weekly demand
# ------------------------------------------------------------

print("ACTIVE WEEKLY SALES DISTRIBUTION:")

print(
    weekly_price_demand["weekly_sales"].describe(
        percentiles=[
            0.50, 0.75, 0.90, 0.95,
            0.99, 0.995, 0.999
        ]
    )
)


# ------------------------------------------------------------
# How extreme is each series' maximum compared with its
# own normal demand?
#
# We use the mean here because many intermittent products
# have median weekly demand = 0.
# ------------------------------------------------------------

weekly_series_stats["max_vs_mean"] = (
    weekly_series_stats["max_weekly_sales"]
    / weekly_series_stats["mean_weekly_sales"].replace(0, np.nan)
)


print("\nMAX / MEAN RATIO ACROSS SERIES:")

print(
    weekly_series_stats["max_vs_mean"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)


# ------------------------------------------------------------
# Show the largest absolute weekly sales observations.
# ------------------------------------------------------------

largest_weeks = (
    weekly_price_demand
    .sort_values("weekly_sales", ascending=False)
    [
        [
            "store_id",
            "item_id",
            "cat_id",
            "state_id",
            "wm_yr_wk",
            "weekly_sales",
            "sell_price",
            "price_vs_typical_pct"
        ]
    ]
    .head(20)
)

print("\n20 LARGEST ITEM-STORE WEEKLY SALES OBSERVATIONS:")

display(
    largest_weeks.round(2)
)


# ------------------------------------------------------------
# Show series with unusually extreme peaks relative to
# their own average.
# ------------------------------------------------------------

print("\nSERIES WITH LARGEST PEAK RELATIVE TO THEIR OWN MEAN:")

display(
    weekly_series_stats
    .sort_values("max_vs_mean", ascending=False)
    .head(20)
    .round(2)
)

In [ ]:
# ============================================================
# SERIES QUALITY / INTERMITTENCY AUDIT
# ============================================================

# We use weekly_price_demand because it already contains only
# active periods according to our frozen first-price rule.

series_quality = (
    weekly_price_demand
    .groupby(["store_id", "item_id", "cat_id", "state_id"])
    .agg(
        active_weeks=("weekly_sales", "size"),
        total_sales=("weekly_sales", "sum"),
        mean_weekly_sales=("weekly_sales", "mean"),
        median_weekly_sales=("weekly_sales", "median"),
        std_weekly_sales=("weekly_sales", "std"),
        zero_weeks=("weekly_sales", lambda x: (x == 0).sum()),
        positive_weeks=("weekly_sales", lambda x: (x > 0).sum())
    )
    .reset_index()
)


# ------------------------------------------------------------
# Intermittency:
# What percentage of active weeks had zero demand?
# ------------------------------------------------------------

series_quality["zero_week_pct"] = (
    series_quality["zero_weeks"]
    / series_quality["active_weeks"]
) * 100


# ------------------------------------------------------------
# Coefficient of variation (CV):
# standard deviation relative to average demand.
#
# Higher CV = demand is more volatile relative to its scale.
# ------------------------------------------------------------

series_quality["cv"] = (
    series_quality["std_weekly_sales"]
    / series_quality["mean_weekly_sales"].replace(0, np.nan)
)


print("ACTIVE HISTORY LENGTH (WEEKS):")

print(
    series_quality["active_weeks"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


print("\nZERO-DEMAND WEEK % PER SERIES:")

print(
    series_quality["zero_week_pct"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


print("\nDEMAND VOLATILITY (CV):")

print(
    series_quality["cv"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


# ------------------------------------------------------------
# Category-level comparison
# ------------------------------------------------------------

category_quality = (
    series_quality
    .groupby("cat_id")
    .agg(
        series_count=("item_id", "size"),
        median_active_weeks=("active_weeks", "median"),
        median_weekly_sales=("mean_weekly_sales", "median"),
        median_zero_week_pct=("zero_week_pct", "median"),
        median_cv=("cv", "median")
    )
    .reset_index()
)


print("\nSERIES QUALITY BY CATEGORY:")

display(
    category_quality.round(2)
)


# ------------------------------------------------------------
# How many series have relatively short histories?
# ------------------------------------------------------------

print("\nSHORT-HISTORY SERIES:")

for weeks in [8, 16, 26, 52, 104]:
    count = (series_quality["active_weeks"] < weeks).sum()

    print(
        f"< {weeks:>3} active weeks: "
        f"{count:>5,} series "
        f"({count / len(series_quality):.2%})"
    )


# ------------------------------------------------------------
# Extremely intermittent series
# ------------------------------------------------------------

print("\nHIGHLY INTERMITTENT SERIES:")

for threshold in [50, 75, 90, 95]:
    count = (
        series_quality["zero_week_pct"] >= threshold
    ).sum()

    print(
        f">= {threshold}% zero weeks: "
        f"{count:>5,} series "
        f"({count / len(series_quality):.2%})"
    )

In [ ]:
# ============================================================
# CELL 25 — HIERARCHY + JOIN INTEGRITY
# ============================================================

# ------------------------------------------------------------
# 1. Verify the product/store hierarchy is consistent.
#
# Each item should always belong to the same department/category.
# Each store should always belong to the same state.
# ------------------------------------------------------------

item_hierarchy = (
    sales.groupby("item_id")
    .agg(
        departments=("dept_id", "nunique"),
        categories=("cat_id", "nunique")
    )
)

store_hierarchy = (
    sales.groupby("store_id")
    .agg(
        states=("state_id", "nunique")
    )
)

print("ITEM HIERARCHY PROBLEMS:")
print(
    "Items linked to multiple departments:",
    (item_hierarchy["departments"] > 1).sum()
)

print(
    "Items linked to multiple categories:",
    (item_hierarchy["categories"] > 1).sum()
)

print("\nSTORE HIERARCHY PROBLEMS:")
print(
    "Stores linked to multiple states:",
    (store_hierarchy["states"] > 1).sum()
)


# ------------------------------------------------------------
# 2. Check expected hierarchy sizes.
# ------------------------------------------------------------

print("\nHIERARCHY COUNTS:")

print("States:", sales["state_id"].nunique())
print("Stores:", sales["store_id"].nunique())
print("Categories:", sales["cat_id"].nunique())
print("Departments:", sales["dept_id"].nunique())
print("Items:", sales["item_id"].nunique())
print("Item-store series:", len(sales))


# ------------------------------------------------------------
# 3. Verify calendar coverage for every sales day.
# ------------------------------------------------------------

sales_day_set = set(day_cols)
calendar_day_set = set(calendar_check["d"])

missing_calendar_days = sales_day_set - calendar_day_set

print("\nCALENDAR JOIN CHECK:")
print(
    "Sales days missing from calendar:",
    len(missing_calendar_days)
)


# ------------------------------------------------------------
# 4. Verify the weekly sales-price join.
#
# weekly_sales contains every item-store-week combination.
# weekly_price_demand contains only combinations with a
# recorded price (our active-period definition).
# ------------------------------------------------------------

print("\nPRICE JOIN CHECK:")

print(
    "Weekly rows before price join:",
    f"{len(weekly_sales):,}"
)

print(
    "Active rows after price join:",
    f"{len(weekly_price_demand):,}"
)

missing_price_rows = (
    len(weekly_sales) - len(weekly_price_demand)
)

print(
    "Rows without recorded price:",
    f"{missing_price_rows:,}"
)

print(
    "Percent excluded as pre-availability:",
    f"{missing_price_rows / len(weekly_sales):.2%}"
)


# ------------------------------------------------------------
# 5. Ensure the price join did NOT duplicate observations.
# ------------------------------------------------------------

duplicate_active_keys = weekly_price_demand.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

print(
    "\nDuplicate item-store-week rows after join:",
    duplicate_active_keys
)


# ------------------------------------------------------------
# 6. Check that all 30,490 series survived.
# ------------------------------------------------------------

active_series_count = (
    weekly_price_demand[
        ["store_id", "item_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "\nSeries remaining after active-period join:",
    f"{active_series_count:,}"
)

print(
    "Original series:",
    f"{len(sales):,}"
)

## 11. Feature Availability and Leakage Contract

Before model training, every feature is classified according to whether it would
actually be available at the time a forecast is generated.

### Static features
These do not change during the forecast horizon:
- item_id
- dept_id
- cat_id
- store_id
- state_id

### Known-future features
These can legitimately be known for future dates:
- date
- weekday / wday
- month
- year
- scheduled event identity and type
- secondary event identity and type
- event proximity features
- the SNAP schedule corresponding to the store's state

### Past-observed features
These may be used only when calculated from information available on or before
the forecast origin:
- historical unit sales
- lag features
- rolling demand statistics
- historical sell prices
- historical price changes
- relative-price features
- product age / time since activation

All lag and rolling features must be shifted appropriately so that future target
values cannot enter model inputs.

### Future price and promotion rule
Future realized prices are not assumed to be known automatically.

A future price or promotion may be used as a known-future feature only when it
represents information explicitly supplied or scheduled by the business at
forecast time.

Historical M5 prices may be used for past-observed price features. Evaluation
must not give the model future realized prices unless that experiment is
explicitly labeled as a planned-price scenario.

### Unknown-future information
The following must never be exposed to the model at forecast time:
- actual future unit sales
- lag or rolling features containing future sales
- unplanned future price changes
- any feature constructed using information unavailable at the forecast origin

This contract applies to both TFT and LightGBM so that their evaluation remains
leakage-safe and operationally realistic.